# Text-to-SQL: QLoRA fine-tuning on Spider

Settings: **Accelerator = GPU T4 x2** (one GPU is used), **Internet = on**.
Inputs: add a Spider dataset under *Add Input* and set `SPIDER_DIR` below to the folder holding `train_spider.json`, `dev.json` and `database/`.
Secrets (*Add-ons -> Secrets*, optional): `HF_TOKEN` to push checkpoints to the Hub, `ANTHROPIC_API_KEY` for the API baseline.

In [ ]:
!pip install -q -U "transformers>=4.56" "trl>=0.20" "peft>=0.13" "bitsandbytes>=0.45" "accelerate>=1.0" "datasets>=3.0" pyyaml anthropic

In [ ]:
# Get the code: clone your repo (or attach it as a Kaggle dataset and copy it here).
!git clone https://github.com/<your-user>/Text_to_SQL.git /kaggle/working/Text_to_SQL
%cd /kaggle/working/Text_to_SQL

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["HF_TOKEN", "ANTHROPIC_API_KEY"]:
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"{name} not set (optional)")

SPIDER_DIR = "/kaggle/input/spider/spider"   # adjust to your dataset path
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
CONFIG = "configs/qwen2.5-coder-1.5b.yaml"
ADAPTER = "/kaggle/working/outputs/qwen2.5-coder-1.5b-spider/final_adapter"

## 1. Prepare data

In [ ]:
!python -m text2sql.prepare_data --spider-dir {SPIDER_DIR} --out-dir data/processed --tokenizer {MODEL}

## 2. Smoke test (a few steps, catches setup errors in ~2 minutes)

In [ ]:
!python -m text2sql.train --config {CONFIG} --max-steps 5 --limit 64 --output-dir /kaggle/working/outputs/smoke

## 3. Full training (add `--resume` to continue after a session timeout)

In [ ]:
!python -m text2sql.train --config {CONFIG}

## 4. Predictions on Spider dev: base vs fine-tuned vs API

In [ ]:
!python -m text2sql.generate --model {MODEL} --out predictions/base.jsonl
!python -m text2sql.generate --model {MODEL} --adapter {ADAPTER} --out predictions/finetuned.jsonl

In [ ]:
# Costs API credits: try --limit 50 first.
!python -m text2sql.generate_api --model claude-opus-5-5 --out predictions/claude-opus-5-5.jsonl

## 5. Execution accuracy

In [ ]:
!python -m text2sql.evaluate predictions/*.jsonl --db-dir {SPIDER_DIR}/database